# 🕸️ Lesson 4.2 — Building a Neural Network: Layers and Depth

**AI Crash Course for Absolute Beginners**

Stack neurons into layers and you get a neural network that can learn any pattern. In this notebook:
- Build a 2-layer network from scratch (pure NumPy)
- Implement forward pass, loss, and backpropagation
- Solve XOR (which the single neuron could not)
- Build a full network with PyTorch and train on real data

> Run each cell with **Shift + Enter**.

In [ ]:
!pip install numpy matplotlib torch torchvision --quiet

import numpy as np
import matplotlib.pyplot as plt

---
## Part 1 — A 2-Layer Network from Scratch

In [ ]:
class TwoLayerNetwork:
    """
    Architecture: input(2) -> hidden(4, ReLU) -> output(1, sigmoid)
    Trained with gradient descent + backprop.
    """

    def __init__(self, input_size, hidden_size, lr=0.1):
        # Xavier initialisation for better gradient flow
        scale1 = np.sqrt(2.0 / input_size)
        scale2 = np.sqrt(2.0 / hidden_size)
        self.W1 = np.random.randn(input_size, hidden_size) * scale1
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, 1) * scale2
        self.b2 = np.zeros((1, 1))
        self.lr = lr

    def relu(self, z):        return np.maximum(0, z)
    def relu_grad(self, z):   return (z > 0).astype(float)
    def sigmoid(self, z):     return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def forward(self, X):
        self.Z1 = X @ self.W1 + self.b1    # hidden pre-activation
        self.A1 = self.relu(self.Z1)        # hidden activation
        self.Z2 = self.A1 @ self.W2 + self.b2  # output pre-activation
        self.A2 = self.sigmoid(self.Z2)         # output probability
        return self.A2

    def backward(self, X, y):
        m = len(y)
        # Output layer gradients
        dZ2 = self.A2 - y.reshape(-1, 1)   # cross-entropy + sigmoid simplification
        dW2 = self.A1.T @ dZ2 / m
        db2 = dZ2.mean(axis=0, keepdims=True)
        # Hidden layer gradients (chain rule)
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self.relu_grad(self.Z1)
        dW1 = X.T @ dZ1 / m
        db1 = dZ1.mean(axis=0, keepdims=True)
        # Gradient descent step
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def fit(self, X, y, epochs=5000):
        losses = []
        for epoch in range(epochs):
            y_hat = self.forward(X).flatten()
            loss  = -np.mean(y * np.log(y_hat + 1e-9) + (1-y) * np.log(1-y_hat + 1e-9))
            self.backward(X, y)
            losses.append(loss)
        return losses

print("TwoLayerNetwork class defined.")

In [ ]:
# Solve XOR
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0,1,1,0], dtype=float)

np.random.seed(42)
net = TwoLayerNetwork(input_size=2, hidden_size=4, lr=0.5)
losses = net.fit(X_xor, y_xor, epochs=5000)

preds = (net.forward(X_xor).flatten() > 0.5).astype(int)
print("XOR predictions after training:")
for xi, yi, pred in zip(X_xor, y_xor, preds):
    ok = "OK" if pred == yi else "WRONG"
    print(f"  {int(xi[0])} XOR {int(xi[1])} = {pred}  (true: {int(yi)})  {ok}")

plt.figure(figsize=(6, 3))
plt.plot(losses, color="#c8401a")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Neural Network Solves XOR")
plt.tight_layout()
plt.show()

---
## Part 2 — PyTorch: The Professional Way

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"PyTorch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# Dataset
X_raw, y_raw = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

# Convert to tensors
X_tr_t = torch.FloatTensor(X_tr).to(device)
y_tr_t = torch.FloatTensor(y_tr).to(device)
X_te_t = torch.FloatTensor(X_te).to(device)
y_te_t = torch.FloatTensor(y_te).to(device)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)
print(f"Train batches: {len(train_loader)}")

In [ ]:
# Define the network
class FeedForwardNet(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super().__init__()
        layers = []
        prev = input_size
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.2)]
            prev = h
        layers += [nn.Linear(prev, output_size)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze()

model = FeedForwardNet(20, [64, 32], 1).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_losses, val_losses = [], []
EPOCHS = 50

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())

    # --- Validation ---
    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_te_t), y_te_t).item()

    train_losses.append(np.mean(batch_losses))
    val_losses.append(val_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}")

In [ ]:
# Plot training curves
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train loss", color="#1a6bc8")
plt.plot(val_losses,   label="Val loss",   color="#c8401a")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Final accuracy
model.eval()
with torch.no_grad():
    preds = (torch.sigmoid(model(X_te_t)) > 0.5).float()
    acc   = (preds == y_te_t).float().mean().item()
print(f"Test accuracy: {acc:.3f}")

---
## Challenge Exercises

1. **Width experiment**: Change hidden sizes from `[64, 32]` to `[256, 128, 64]`. Does accuracy improve? Does training slow?
2. **Dropout tuning**: Change `Dropout(0.2)` to `Dropout(0.5)`. How does the train/val gap change?
3. **Learning rate**: Try `lr=0.01` and `lr=0.0001`. Compare loss curves.

---
**Next lesson:** [Lesson 4.3 — Convolutional Neural Networks](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-4.3-cnn.ipynb)